# 3장 실습 — 최단경로 직접 짜기 (채점)

이번 학기에 직접 짜는 셋 중 첫 번째입니다.

채울 파일은 **`labs/ch03_dijkstra.py`** 입니다. 이 노트북이 아니라 그 파일을 고칩니다.
노트북은 채운 것을 바로 확인하는 용도입니다. 편집기에서 파일을 열어 두고,
함수 하나를 채울 때마다 이 노트북의 해당 셀을 다시 실행합니다.

순서는 이렇습니다.

1. 손으로 답을 아는 노드 6개짜리 그래프에서 확인합니다 (교재 3.2)
2. 하남시 도로망으로 옮깁니다 (교재 3.3)
3. NetworkX 와 대조합니다 (교재 3.4)
4. A\* 를 더해 확정 노드 수와 실행 시간을 잽니다 (교재 3.5, 3.6)
5. `check("ch03")` 으로 채점합니다

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect
from smartmob.viz import use_korean_font

use_korean_font()

import ch03_dijkstra as sol      # 여러분이 채우는 파일

`autoreload` 를 켜 두었으므로 `labs/ch03_dijkstra.py` 를 저장하면
이 노트북을 다시 시작하지 않아도 반영됩니다. `sol` 이 그 파일입니다.

## 1. 손으로 답을 아는 그래프 (교재 3.2)

노드 여섯 개짜리 그래프를 만듭니다. 교재 3.2절의 그림과 같은 것입니다.

```
       (600m)        (900m)
  A ────────────► B ────────────► C
  │               │               ▲
  │(2000m)        │(300m)         │(300m)
  │               ▼               │
  └──────────────►D───────────────┘

  E ────────────► F        (A 쪽과 이어져 있지 않습니다)
         (300m)
```

모든 도로가 시속 36km, 즉 초속 10m입니다. 그래서 거리를 10으로 나누면 초가 됩니다.

A에서 C로 가는 길은 셋입니다.

| 경로 | 거리 | 소요시간 |
|---|---|---|
| A → C | 2,000m | 200초 |
| A → B → C | 1,500m | 150초 |
| A → B → D → C | 1,200m | **120초** |

정답은 120초, 경로는 `[n1, n2, n4, n3]` 입니다.
교재 3.2절의 표는 이 그래프에서 힙에서 꺼내는 순서를 한 줄씩 적은 것입니다.
A, B, D, C 순서로 네 개를 꺼내면 끝나므로 확정한 노드는 4개입니다.

노드 표와 엣지 표를 손으로 만들어 `RoadGraph.from_frames` 에 넘깁니다.
`edge_id` 는 2장의 규칙대로 `e1_f_1_2` 가 "노드 n1 에서 n2 로" 입니다.

In [ ]:
import pandas as pd

from smartmob.teaching.graph import RoadGraph

toy_nodes = pd.DataFrame([
    {"node_id": "n1", "lat": 37.5000, "lon": 127.2000},   # A
    {"node_id": "n2", "lat": 37.5030, "lon": 127.2000},   # B
    {"node_id": "n3", "lat": 37.5080, "lon": 127.2000},   # C
    {"node_id": "n4", "lat": 37.5050, "lon": 127.2010},   # D
    {"node_id": "n5", "lat": 37.5500, "lon": 127.2500},   # E
    {"node_id": "n6", "lat": 37.5520, "lon": 127.2500},   # F
])

toy_edges = pd.DataFrame([
    {"edge_id": "e1_f_1_2", "length": 600.0},     # A → B
    {"edge_id": "e2_f_1_3", "length": 2000.0},    # A → C
    {"edge_id": "e3_f_2_3", "length": 900.0},     # B → C
    {"edge_id": "e4_f_2_4", "length": 300.0},     # B → D
    {"edge_id": "e5_f_4_3", "length": 300.0},     # D → C
    {"edge_id": "e6_f_5_6", "length": 300.0},     # E → F
]).assign(highway="residential", free_flow_speed_kmh=36.0)   # 모든 엣지에 같은 종류와 속도를 붙입니다

toy = RoadGraph.from_frames(toy_nodes, toy_edges, modes=("drive",))
toy

출력에 노드 6개, 엣지 6개가 나오면 됩니다.

이제 `labs/ch03_dijkstra.py` 의 `dijkstra` 와 `trace` 를 채웁니다. 각 함수의 docstring 에 힌트가 있습니다.
`heapq` 는 리스트를 최소 힙으로 다루는 모듈입니다.
`heappush(heap, (거리, 노드))` 로 넣고 `heappop(heap)` 으로 꺼내면 거리가 가장 작은 튜플이 먼저 나옵니다.
교재 3.2절 표의 "꺼낸 것" 열이 이 `heappop` 의 결과입니다.

아직 안 채웠으면 "아직 구현하지 않았습니다"가 나옵니다. 정상입니다.

In [ ]:
banner("작은 그래프 확인")
try:
    seconds, path, settled = sol.dijkstra(toy, "n1", "n3")
    expect("A→C 소요시간(초)", seconds, 120.0, tol=0.01)
    expect("A→C 경로", path, ["n1", "n2", "n4", "n3"])
    expect("확정한 노드 수", settled, 4)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)

길이 없는 경우도 확인합니다. A에서 E로는 갈 수 없으므로 예외가 나야 합니다.
교재는 `NoPath` 라는 예외 클래스를 정의해 던집니다. `ValueError` 처럼 다른 예외를 던져도 채점은 통과합니다.
조용히 무한대를 돌려주면 시뮬레이터가 그 승객을 영원히 기다리게 만듭니다.

In [ ]:
try:
    sol.dijkstra(toy, "n1", "n5")
    print("[x] 예외를 던지지 않았습니다. 길이 없을 때는 예외를 내야 합니다")
except NotImplementedError:
    print("[ ] 아직 구현하지 않았습니다")
except Exception as exc:
    print(f"[v] 길이 없을 때 {type(exc).__name__} 를 냅니다 — {exc}")

## 2. 하남시 도로망으로 (교재 3.1, 3.3)

작은 그래프에서 맞으면 실제 도로망으로 옮깁니다. 노드가 12,566개로 늘어날 뿐 알고리즘은 그대로입니다.
하남시청과 미사역의 좌표를 `nearest_node` 로 가장 가까운 노드에 붙입니다. 두 지점의 직선거리는 3.05km입니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))
start = G.nearest_node(37.5393, 127.2148)    # 하남시청
goal = G.nearest_node(37.5606, 127.1930)     # 미사역

print("출발", start, G.coord[start])
print("도착", goal, G.coord[goal])

출발과 도착이 노드 id 로 바뀌었습니다.
같은 함수를 이 두 노드 사이에서 돌리고, 경로 노드의 좌표를 이어 도로 길이도 구합니다.

In [ ]:
from smartmob.teaching.graph import haversine_km

try:
    seconds, path, settled = sol.dijkstra(G, start, goal)
    # 경로에서 이웃한 노드 쌍마다 직선거리를 더해 도로 길이를 어림합니다
    km = sum(haversine_km(*G.coord[u], *G.coord[v]) for u, v in zip(path, path[1:]))
    print(f"소요시간 {seconds / 60:.2f}분   경로 길이 {km:.1f}km")
    print(f"거친 노드 {len(path):,}개")
    print(f"확정한 노드 {settled:,}개  (전체 {G.n_nodes:,}개의 {settled / G.n_nodes:.0%})")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

교재와 같은 값이 나와야 합니다. 소요시간 5.54분, 경로 길이 4.2km, 경로 노드 83개, 확정 노드 4,565개입니다.
직선 3km 자리를 도로로 4.2km 달렸고, 자유류 속도라 신호 대기는 빠져 있습니다.

눈여겨볼 것은 마지막 줄입니다. 83개 노드짜리 경로를 얻으려고 전체의 36%인 4,565개를 확정했습니다.
출발점에서 사방으로 고르게 퍼지기 때문입니다. 이것이 5절에서 A\*를 도입하는 이유입니다.

## 3. 경로가 실제로 이어져 있는지 (교재 3.3)

소요시간이 맞아도 `prev` 를 잘못 채우면 경로가 끊어져 있을 수 있습니다.
경로의 이웃한 노드 쌍마다 그 엣지를 찾아 초를 다시 더하고, 반환값과 같은지 봅니다.
엣지가 없으면 `assert` 가 멈춥니다.

In [ ]:
try:
    seconds, path, settled = sol.dijkstra(G, start, goal)
    total = 0.0
    for u, v in zip(path, path[1:]):
        # u 의 이웃 중 v 를 찾아 그 엣지의 초를 꺼냅니다. 없으면 None 입니다
        edge = next((w for nb, w, _ in G.neighbors(u) if nb == v), None)
        assert edge is not None, f"{u} → {v} 엣지가 없습니다"
        total += edge
    expect("엣지를 다시 더한 값", round(total, 3), round(seconds, 3), tol=0.01)
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 4. NetworkX 와 대조 (교재 3.4)

코드가 실행되는 것과 결과가 맞는 것은 별개입니다. 남이 만든 구현과 같은 답이 나오는지 봅니다.
먼저 우리 그래프를 NetworkX 의 방향 그래프 `DiGraph` 로 옮깁니다.
같은 두 노드 사이에 엣지가 여럿이면 가장 짧은 것 하나만 남깁니다. 최단경로는 어차피 그것만 쓰기 때문입니다.

In [ ]:
import random

import networkx as nx

nxg = nx.DiGraph()
for u in G.nodes:
    for v, seconds_, _ in G.neighbors(u):
        # 이미 있는 엣지보다 짧을 때만 덮어씁니다
        if not nxg.has_edge(u, v) or nxg[u][v]["weight"] > seconds_:
            nxg.add_edge(u, v, weight=seconds_)

print(f"NetworkX 그래프 노드 {nxg.number_of_nodes():,}개, 엣지 {nxg.number_of_edges():,}개")

엣지가 28,109개로 우리 그래프의 28,589개보다 적습니다.
같은 노드 쌍을 잇는 엣지 480개가 하나로 합쳐졌기 때문입니다.

무작위로 30쌍을 뽑아 최단 소요시간을 맞춰 봅니다. `python labs/check.py ch03` 도 같은 방식으로 30쌍을 봅니다.
무작위 쌍의 6% 남짓은 길이 아예 없으므로, NetworkX 가 `NetworkXNoPath` 를 내면 그 쌍은 건너뜁니다.

In [ ]:
rng = random.Random(42)
node_list = [n for n in G.nodes if G.adj[n]]     # 나가는 엣지가 있는 노드만

banner("NetworkX 대조 (30쌍)")
try:
    worst, compared = 0.0, 0
    while compared < 30:
        u, v = rng.choice(node_list), rng.choice(node_list)
        try:
            want = nx.shortest_path_length(nxg, u, v, weight="weight")
        except nx.NetworkXNoPath:
            continue                                  # 길이 없는 쌍은 건너뜁니다
        got, _, _ = sol.dijkstra(G, u, v)
        worst = max(worst, abs(got - want))
        compared += 1
    print(f"{compared}쌍 비교, 최대 오차 {worst:.6f}초")
    print("[v] 통과" if worst < 1e-6 else "[x] 값이 다릅니다. 도착 노드를 힙에서 꺼낼 때 끝냈는지 확인합니다")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 5. A\* (교재 3.5, 3.6)

다익스트라가 사방으로 퍼지는 것을 목적지 쪽으로 밀어 줍니다.
힌트 `h(n)` 은 남은 직선거리를 이 도로망의 최고 속도로 달리는 시간입니다.
실제 남은 시간보다 항상 작으므로(허용 가능) 최적해가 유지됩니다.

`labs/ch03_dijkstra.py` 의 `astar` 를 채우고 실행합니다.
힙에 넣는 우선순위는 `이미 온 시간 + h(n)` 이고, 꺼낸 뒤 이웃을 갱신할 때 쓰는 것은 `이미 온 시간` 입니다.

In [ ]:
print(f"이 도로망의 최고 속도 {G.max_speed_kmh():.0f} km/h")

banner("다익스트라와 A* 비교")
try:
    d_sec, d_path, d_settled = sol.dijkstra(G, start, goal)
    a_sec, a_path, a_settled = sol.astar(G, start, goal)

    expect("두 방법의 소요시간이 같다", round(a_sec, 6), round(d_sec, 6), tol=1e-6)
    print(f"    확정 노드  다익스트라 {d_settled:,}  →  A* {a_settled:,}"
          f"  ({1 - a_settled / d_settled:.0%} 감소)")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

소요시간은 같고 확정 노드는 4,565개에서 3,571개로 22% 줄어듭니다.
교재 3.6절의 "1.7배"는 무작위 쌍 30개의 중앙값이라 값이 다릅니다.
쌍 하나로 알고리즘을 판단하면 안 된다는 뜻이기도 합니다.

확정 노드는 줄어드는데 실행 시간은 오히려 A\*가 길 수 있습니다. 같은 질의를 다섯 번 돌려 평균을 잽니다.

In [ ]:
import time

try:
    for name, fn in [("다익스트라", sol.dijkstra), ("A*", sol.astar)]:
        t0 = time.perf_counter()
        for _ in range(5):
            fn(G, start, goal)
        print(f"{name:10s} {(time.perf_counter() - t0) / 5 * 1000:7.1f} ms")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

교재를 만든 컴퓨터에서는 다익스트라 중앙값이 7.4ms 였고 A\* 가 그보다 느렸습니다.
컴퓨터마다 값은 다르지만 순서는 대개 같습니다.
이유는 `h(n)` 입니다. 노드를 꺼낼 때마다 하버사인 거리를 계산하는데,
삼각함수가 네 번 들어가는 계산이 파이썬에서는 힙 연산보다 비쌉니다.
노드를 22% 덜 보는 대신 보는 노드마다 일을 더 하니 상쇄되고도 남습니다.

## 6. 정리된 코드 (교재 3.8)

같은 알고리즘이 `smartmob.teaching.dijkstra` 에 들어 있고 4장부터는 이것을 씁니다.
반환값이 다릅니다. 튜플 셋 대신 `Path` 객체 하나를 돌려주고, `duration_s`, `nodes`, `settled` 를 속성으로 꺼냅니다.
길이 없으면 `NoPath` 를 던집니다. 내 구현과 소요시간이 같은지 맞춰 봅니다.

In [ ]:
from smartmob.teaching.dijkstra import NoPath, dijkstra as lib_dijkstra

p = lib_dijkstra(G, start, goal)
print(f"라이브러리 판  {p.duration_min:.2f}분, 경로 노드 {len(p.nodes)}개, 확정 {p.settled:,}개")

try:
    seconds, path, settled = sol.dijkstra(G, start, goal)
    expect("내 구현과 소요시간이 같다", round(seconds, 6), round(p.duration_s, 6), tol=1e-6)
    print(f"    확정 노드  내 구현 {settled:,}  /  라이브러리 {p.settled:,}")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 7. 채점

과제로 제출할 때 보는 것과 같은 기준입니다. 다섯 항목이 전부 PASS 면 됩니다.

In [ ]:
from check import check

report = check("ch03")

## 제출할 것

1. 채운 `labs/ch03_dijkstra.py`
2. 위 채점 셀의 출력 (전부 PASS)
3. 막혔던 지점과 어떻게 풀었는지 3~5줄

## 정리

- 다익스트라는 "미확정 노드 중 가장 가까운 것은 이미 최종 답이다"를 반복합니다. 엣지 비용이 음수가 아니어야 성립합니다
- 도착 노드를 힙에 **넣을 때**가 아니라 **꺼낼 때** 끝냅니다
- 길이 없으면 예외를 냅니다. 무한대를 조용히 돌려주면 시뮬레이터가 망가집니다
- 하남시청→미사역은 5.54분, 경로 노드 83개, 확정 노드 4,565개입니다
- A\*는 확정 노드를 줄이지만 파이썬에서는 더 느릴 수 있습니다
- 4장 실습에서는 같은 그래프에 시간대별 속도를 넣어 경로가 어떻게 바뀌는지 봅니다